# PCG Accelerator Test

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
bitstream_path = '/home/xilinx/pcg/pcg.bit'
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Problem Setup & Matrix Generation
# =====================================================================
print("\nRunning PCG CSC testbench...")

num_rows = 1024
num_cols = 1024
PACK_SIZE = 16 
MAX_NNZ_WORDS = (1024 * 1024) // PACK_SIZE

sigma = 0.1
epsilon_sq = 1e-8

np.random.seed(42)

print(f"Generating A({num_rows}x{num_cols}) and P({num_cols}x{num_cols}) matrices...")
A_sparse = sp.random(num_rows, num_cols, density=1.0/num_rows, format='csc', dtype=np.float32, random_state=42)

A_values_data = A_sparse.data
A_values_data[np.abs(A_values_data) < 1e-3] = 1e-3
A_sparse.data = A_values_data

A_row_idx_data = A_sparse.indices.astype(np.int32)
A_col_ptr_data = A_sparse.indptr.astype(np.int32)
A_nnz = A_sparse.nnz

AT_sparse = A_sparse.transpose().tocsc()
AT_values_data = AT_sparse.data.astype(np.float32)
AT_row_idx_data = AT_sparse.indices.astype(np.int32)
AT_col_ptr_data = AT_sparse.indptr.astype(np.int32)

P_values_data = np.random.uniform(1.0, 2.0, num_cols).astype(np.float32)
P_row_idx_data = np.arange(num_cols, dtype=np.int32)
P_col_ptr_data = np.arange(num_cols + 1, dtype=np.int32)
P_nnz = num_cols

b_data = np.random.uniform(-1.0, 1.0, num_cols).astype(np.float32)
rho_data = np.ones(num_rows, dtype=np.float32)

M_inv_data = np.zeros(num_cols, dtype=np.float32)
for c in range(num_cols):
    diagK = P_values_data[c] + sigma
    start, end = A_col_ptr_data[c], A_col_ptr_data[c+1]
    for idx in range(start, end):
        r = A_row_idx_data[idx]
        v = A_values_data[idx]
        diagK += rho_data[r] * (v ** 2)
    M_inv_data[c] = 1.0 / diagK

# =====================================================================
# 3. Buffer Allocation and Packing
# =====================================================================
def pack_csc_to_words(row_idx, values):
    nnz = len(row_idx)
    
    # FIX 1: Use int32 and float32 so that 16 elements = exactly 512 bits
    # FIX 2: Use cacheable=False to bypass Zynq ARM cache sync issues
    row_words = allocate(shape=(MAX_NNZ_WORDS, PACK_SIZE), dtype=np.int32, cacheable=False)
    val_words = allocate(shape=(MAX_NNZ_WORDS, PACK_SIZE), dtype=np.float32, cacheable=False)
    
    row_words[:] = 0
    val_words[:] = 0.0
    
    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values
    return row_words, val_words

print("Allocating and Packing Hardware CMA Buffers (Uncached)...")

A_row_words, A_val_words = pack_csc_to_words(A_row_idx_data, A_values_data)
AT_row_words, AT_val_words = pack_csc_to_words(AT_row_idx_data, AT_values_data)
P_row_words, P_val_words = pack_csc_to_words(P_row_idx_data, P_values_data)

A_col_ptr_hw  = allocate(shape=(num_cols + 1,), dtype=np.int32, cacheable=False); A_col_ptr_hw[:]  = A_col_ptr_data
AT_col_ptr_hw = allocate(shape=(num_rows + 1,), dtype=np.int32, cacheable=False); AT_col_ptr_hw[:] = AT_col_ptr_data
P_col_ptr_hw  = allocate(shape=(num_cols + 1,), dtype=np.int32, cacheable=False); P_col_ptr_hw[:]  = P_col_ptr_data

b_hw     = allocate(shape=(num_cols,), dtype=np.float32, cacheable=False); b_hw[:]     = b_data
rho_hw   = allocate(shape=(num_rows,), dtype=np.float32, cacheable=False); rho_hw[:]   = rho_data
M_inv_hw = allocate(shape=(num_cols,), dtype=np.float32, cacheable=False); M_inv_hw[:] = M_inv_data
x_out_hw = allocate(shape=(num_cols,), dtype=np.float32, cacheable=False); x_out_hw[:] = 0.0

# =====================================================================
# 4. Hardware Execution
# =====================================================================
CTRL_AP_CTRL    = 0x00
CTRL_NUM_ROWS   = 0x10
CTRL_NUM_COLS   = 0x18
CTRL_A_NNZ      = 0x20
CTRL_P_NNZ      = 0x28
CTRL_SIGMA      = 0x30
CTRL_EPSILON_SQ = 0x38

CTRL_R_A_ROW    = 0x10
CTRL_R_A_COL    = 0x1c
CTRL_R_A_VAL    = 0x28
CTRL_R_AT_ROW   = 0x34
CTRL_R_AT_COL   = 0x40
CTRL_R_AT_VAL   = 0x4c
CTRL_R_P_ROW    = 0x58
CTRL_R_P_COL    = 0x64
CTRL_R_P_VAL    = 0x70
CTRL_R_M_INV    = 0x7c
CTRL_R_B        = 0x88
CTRL_R_RHO      = 0x94
CTRL_R_X_OUT    = 0xa0

def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', f_val))[0]

print("Configuring Registers...")

control_ip.write(CTRL_NUM_ROWS, num_rows)
control_ip.write(CTRL_NUM_COLS, num_cols)
control_ip.write(CTRL_A_NNZ, A_nnz)
control_ip.write(CTRL_P_NNZ, P_nnz)
control_ip.write(CTRL_SIGMA, float_to_uint(sigma))
control_ip.write(CTRL_EPSILON_SQ, float_to_uint(epsilon_sq))

write_64bit_address(control_r_ip, CTRL_R_A_ROW, A_row_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_A_COL, A_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_A_VAL, A_val_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_AT_ROW, AT_row_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_AT_COL, AT_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_AT_VAL, AT_val_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_P_ROW, P_row_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_P_COL, P_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_P_VAL, P_val_words.device_address)
write_64bit_address(control_r_ip, CTRL_R_M_INV, M_inv_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_B, b_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_RHO, rho_hw.device_address)
write_64bit_address(control_r_ip, CTRL_R_X_OUT, x_out_hw.device_address)

print("Starting Hardware Accelerator...")
hw_start = time.time()

control_ip.write(CTRL_AP_CTRL, 0x01) 
while (control_ip.read(CTRL_AP_CTRL) & 0x02) == 0:
    pass

hw_end = time.time()
print(f"HW Execution Time: {(hw_end - hw_start) * 1000:.4f} ms")

x_hw = np.array(x_out_hw)

# =====================================================================
# 5. Software Reference & Verification
# =====================================================================
print("\nRunning CPU Reference Check...")

tmp0 = A_sparse.dot(x_hw)
tmp1 = rho_data * tmp0
tmp2 = AT_sparse.dot(tmp1)

P_sparse = sp.csc_matrix((P_values_data, P_row_idx_data, P_col_ptr_data), shape=(num_cols, num_cols))
px = P_sparse.dot(x_hw)

Kx = tmp2 + px + (sigma * x_hw)

norm_b2 = np.sum(b_data ** 2)
norm_r2 = np.sum((Kx - b_data) ** 2)
rel_res = np.sqrt(norm_r2 / (norm_b2 + 1e-30))

print("\n--- Results ---")
print(f"||b||^2    = {norm_b2:.6e}")
print(f"||Kx-b||^2 = {norm_r2:.6e}")
print(f"rel_res    = {rel_res:.6e}")

target_res = np.sqrt(epsilon_sq)
if rel_res <= 5.0 * target_res:
    print(f">>> SUCCESS: PCG Hardware residual meets tolerance (~{target_res:.1e})! <<<")
else:
    print(f">>> ERROR: Residual too large (Target ~ {target_res:.1e}). <<<")

for buf in [A_row_words, A_val_words, AT_row_words, AT_val_words, P_row_words, P_val_words,
            A_col_ptr_hw, AT_col_ptr_hw, P_col_ptr_hw, b_hw, rho_hw, M_inv_hw, x_out_hw]:
    buf.freebuffer()